In [ ]:
import os
import copy
import random

import matplotlib.pyplot as plt
import numpy as np
import torch

from src.config import DL_CONFIG
from src.data import load_and_prep_data_strided
from src.gpu_backtest import run_multigpu_backtest
from src.metrics import calculate_global_metrics

random.seed(42)

CONFIG = copy.deepcopy(DL_CONFIG)
CONFIG['data_path'] = 'all30min/'

LOADING_HPARAMS = {
    'exog_cols': 'none',
    'use_transform_exog': False,
    'use_diurnal': True,
    'allow_missing': False,
    'use_winsor': False,
}

torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'

In [ ]:
# Load data using codebase pipeline with dense consecutive lags for the DL model
X_np, y_np, dates, baselines, features = load_and_prep_data_strided(
    LOADING_HPARAMS,
    CONFIG['data_path'],
    lag=CONFIG['model']['context_len']
)
print(f'Data: {X_np.shape[0]} samples, {X_np.shape[1]} features')

In [ ]:
# Run GPU backtest (uses src.dl_models.PatchTSMixerForecaster by default)
# Swap model_module for a different architecture in future notebooks
results = run_multigpu_backtest(
    X_np, y_np, dates, baselines, CONFIG,
    model_module='src.dl_models'
)

In [ ]:
# Evaluate using codebase metrics
metrics = calculate_global_metrics(results)
print(f"QLIKE: {metrics.get('qlike', float('nan')):.4f}")
print(f"MSE:   {metrics.get('mse', float('nan')):.6f}")
print(f"MAE:   {metrics.get('mae', float('nan')):.6f}")

# Time series plot
plt.figure(figsize=(12, 6))
subset = results.iloc[-500:]
plt.plot(subset.index, subset['true_raw'], label='True RV', alpha=0.6, color='black')
plt.plot(subset.index, subset['pred_raw'], label='Predicted RV', alpha=0.8, color='#1f77b4')
plt.title('Volatility Forecast: Model vs Ground Truth')
plt.ylabel('Realized Volatility')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Diagnostic scatter
plt.figure(figsize=(10, 10))
plt.scatter(results['true_raw'], results['pred_raw'], alpha=0.3, s=10, label='Predictions')
plt.plot([1e-6, 1e-2], [1e-6, 1e-2], 'k-', alpha=0.5, label='Perfect Match')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('True Volatility')
plt.ylabel('Predicted Volatility')
plt.title(f"Diagnostic: QLIKE = {metrics.get('qlike', float('nan')):.4f}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()